# Master de Inteligencia Artificial

**Alumno:** José Ignacio Castillo Camacho · **Matrícula:** AL07161745

**Actividad 6.** Evaluación y validación del sistema inteligente de alertas de sobrecosto: VOREKO IA

## 1. Introducción a VOREKO y continuidad con la Actividad 5

## 2. Carga del dataset versionado

In [ ]:
import os, subprocess, sys
from pathlib import Path
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    REPO = Path("/content/voreko-ia-incidencias")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Ignacio-Castillo-C/voreko-ia-incidencias.git", str(REPO)], check=True)
    os.chdir(REPO / "Actividad6" / "fuentes")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow>=3.0"], check=True)
for carpeta in [Path.cwd(), Path.cwd() / "fuentes", Path.cwd() / "Actividad6" / "fuentes"]:
    if (carpeta / "voreko_a6.py").exists():
        sys.path.insert(0, str(carpeta))
        break
else:
    raise FileNotFoundError("No se encontró voreko_a6.py")

In [ ]:
import json, platform, time, warnings
from datetime import datetime

import matplotlib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import scipy, sklearn, xgboost
from scipy import stats
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve, roc_auc_score, roc_curve
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold

import voreko_a6 as v
%matplotlib inline

warnings.filterwarnings("ignore")
SEMILLA = v.SEMILLA
np.random.seed(SEMILLA)
RES = v.RAIZ_A6 / "resultados"
FIG, TAB = RES / "figuras", RES / "tablas"
for d in (FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)
FECHA = datetime.now().strftime("%Y-%m-%d %H:%M")
VERSIONES = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
             "xgboost": xgboost.__version__, "scipy": scipy.__version__, "mlflow": mlflow.__version__, "matplotlib": matplotlib.__version__}
print("Ejecución:", FECHA, "|", VERSIONES)

COLORES = {"Dummy (baseline)": "#8a8a85", "Regla A5 (XGBoost regresión)": "#c98500", "Regresión logística": "#2a78d6",
           "Random Forest": "#eb6834", "XGBoost clasificador": "#1baf7a"}
CORTO = {"Dummy (baseline)": "Dummy", "Regla A5 (XGBoost regresión)": "Regla A5", "Regresión logística": "Reg. logística",
         "Random Forest": "Random Forest", "XGBoost clasificador": "XGBoost clf."}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e3",
                     "axes.axisbelow": True, "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False})

def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=200, bbox_inches="tight")
    plt.show()

CHECKS = []
def verificar(cond, texto):
    assert cond, f"FALLÓ: {texto}"
    CHECKS.append(texto); print("✔", texto)

df, manifiesto_a5 = v.cargar_datos_a5()
verificar(v.sha256(v.dp.CSV_LIMPIO) == manifiesto_a5["huellas_sha256"]["csv_limpio"], "Huella SHA-256 del dataset limpio de la A5 = manifiesto")
verificar(df.shape == (372, 117), "Dataset de la A5: 372 obras × 117 columnas")
print("Versión de datos A5:", manifiesto_a5["version_datos"], "| fuente:", manifiesto_a5["fuente"])

## 3. Construcción de la variable objetivo

In [ ]:
desv = v.desviacion_real(df)
y = v.construir_etiqueta(df)
empates = int((desv == 0.5).sum())
flotante = int((((df.costo_real_m2 - df.costo_prelim_m2) / df.costo_prelim_m2) > 0.5).sum())
print(f"Obras con desviación exactamente 50 %: {empates} → clase 0. Sin redondeo se contarían {flotante} positivas en lugar de {int(y.sum())}.")
man_clasif = v.guardar_version_clasificacion(df, manifiesto_a5)
verificar(v.sha256(v.dp.CSV_LIMPIO) == v.SHA_A5_ESPERADO, "El dataset de la A5 sigue intacto tras crear la versión derivada")
print(json.dumps({k: man_clasif[k] for k in ["version_datos", "sha256_csv_clasificacion", "positivos", "prevalencia"]}, indent=1))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(desv * 100, bins=np.arange(-25, 405, 10), color="#86b6ef", edgecolor="white")
ax[0].axvline(50, color="#c0392b", lw=1.5); ax[0].text(53, ax[0].get_ylim()[1] * 0.9, "umbral de sobrecosto 50 %", color="#c0392b", fontsize=9)
ax[0].set_xlabel("Desviación real del costo por m² (%)"); ax[0].set_ylabel("Obras"); ax[0].set_title("(a) Desviación real costo real vs. preliminar")
c = y.value_counts().sort_index()
ax[1].bar(["Clase 0 (≤ 50 %)", "Clase 1 (> 50 %)"], c.values, color=["#b7d3f6", "#2a78d6"], width=0.5)
for i, n in enumerate(c.values):
    ax[1].text(i, n, f"{n} ({n / len(y):.1%})", ha="center", va="bottom")
ax[1].set_ylim(0, c.max() * 1.15); ax[1].set_title("(b) riesgo_sobrecosto"); ax[1].set_ylabel("Obras")
fig.tight_layout(); guardar(fig, "01_variable_objetivo")

## 4. Análisis de clases

In [ ]:
sens_prev = pd.DataFrame([{"umbral_sobrecosto": f"{int(u * 100)} %", "positivos": int(v.construir_etiqueta(df, u).sum()),
                           "prevalencia": round(float(v.construir_etiqueta(df, u).mean()), 3),
                           "obras_en_el_limite": int((desv == u).sum())} for u in (0.2, 0.3, 0.4, 0.5)])
display(sens_prev)
por_anio = pd.crosstab(df.anio_inicio, y).rename(columns={0: "clase_0", 1: "clase_1"})
por_anio["tasa_riesgo"] = (por_anio.clase_1 / por_anio.sum(axis=1)).round(3)
display(por_anio.T)
sens_prev.to_csv(TAB / "prevalencia_por_umbral_sobrecosto.csv", index=False)
por_anio.to_csv(TAB / "riesgo_por_anio_inicio.csv")

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(por_anio.index.astype(str), por_anio.tasa_riesgo * 100, color="#2a78d6", width=0.6)
for i, (t, n) in enumerate(zip(por_anio.tasa_riesgo, por_anio.clase_0 + por_anio.clase_1)):
    ax.text(i, t * 100 + 1.5, f"n={n}", ha="center", fontsize=8, color="#5f5e58")
ax.axhline(y.mean() * 100, color="#8a8a85", ls="--", lw=1, label=f"Promedio {y.mean():.1%}")
ax.set_xlabel("Año de inicio (calendario persa)"); ax.set_ylabel("% de obras con sobrecosto > 50 %")
ax.set_title("Tasa de riesgo por año de inicio: la prevalencia cambia en el tiempo"); ax.legend()
fig.tight_layout(); guardar(fig, "02_riesgo_por_anio")

## 5. Preparación y validación de datos

In [ ]:
dur = v.verificar_duracion_real(df)
print(dur)
verificar(dur["fin_igual_inicio_mas_duracion"] == 372, "En las 372 obras: término real = inicio + duración → la duración es real, no planeada")
X = v.variables_inicio(df)
prohibidas = {"costo_real_m2", "precio_venta_real_m2", "anio_fin", "trimestre_fin", "indice_tiempo_fin", "anio_fin_greg_aprox",
              "razon_real_vs_preliminar", "costo_prelim_total", "duracion_trimestres", "anomalia_costo_m2", "anomalia_razon_costo"}
verificar(not prohibidas & set(X.columns), "Ninguna variable posterior al inicio (ni la duración real) entra a los modelos")
verificar(X.isna().sum().sum() == 0 and not X.duplicated().any(), "Sin nulos ni filas duplicadas en las predictoras")
r = manifiesto_a5["reporte"]
calidad = {"predictoras_A5": manifiesto_a5["predictoras"], "predictoras_A6": X.shape[1], "excluida_A6": list(v.EXCLUIDAS_A6),
           "duplicados_exactos_A5": r["4_duplicados"]["duplicados_exactos"], "violaciones_reglas_A5": r["5_reglas_negocio"]["filas_eliminadas"],
           "anomalias_razon_marcadas_A5": r["7_anomalias"]["anomalias_razon_costo"]}
print(calidad)

## 6. Separación entrenamiento/prueba

In [ ]:
idx_tr, idx_te = v.particion_a5(X, df["costo_real_m2"])
X_tr, X_te, y_tr, y_te = X.loc[idx_tr], X.loc[idx_te], y.loc[idx_tr], y.loc[idx_te]
c_tr, c_te = df.loc[idx_tr, "costo_real_m2"], df.loc[idx_te, "costo_real_m2"]
verificar(len(X_tr) == 297 and len(X_te) == 75, "Partición 297/75 idéntica en tamaño a la Actividad 5")
verificar(len(set(df.loc[idx_tr, "id_proyecto"]) & set(df.loc[idx_te, "id_proyecto"])) == 0, "Sin obras compartidas entre entrenamiento y prueba")
particion = pd.DataFrame({"obras": [len(y_tr), len(y_te)], "riesgo=1": [int(y_tr.sum()), int(y_te.sum())],
                          "prevalencia": [round(y_tr.mean(), 3), round(y_te.mean(), 3)]}, index=["Entrenamiento", "Prueba"])
particion.to_csv(TAB / "particion.csv"); display(particion)
CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
print("Positivos por pliegue de validación (5 pliegues):", [int(y_tr.iloc[va].sum()) for _, va in CV5.split(X_tr, y_tr)])

## 7. Baseline

In [ ]:
ajustados, s_te, s_oof = {}, {}, {}
n = "Dummy (baseline)"
ajustados[n] = v.ajustar(n, X_tr, y_tr, c_tr)
s_te[n] = v.puntuar(ajustados[n], X_te)
s_oof[n] = v.oof(n, v.fijo(n, ajustados[n]), X_tr, y_tr, c_tr, CV5)
print(f"Probabilidad constante del baseline: {s_te[n][0]:.3f}")

## 8. Reutilización del enfoque XGBoost de regresión (Actividad 5)

In [ ]:
X_a5, _ = v.dp.seleccionar_variables(df)
regla_hist = v.ReglaRegresionA5().fit(X_a5.loc[idx_tr], c_tr)
hist = v.evaluar(y_te, regla_hist.puntuar(X_a5.loc[idx_te]), 1.5)
reportado_a5 = {"recall": 0.692, "precision": 0.783, "roc_auc": 0.859}
print({k: round(hist[k], 3) for k in ["recall", "precision", "roc_auc", "average_precision", "VP", "FP", "FN"]})
verificar(all(abs(hist[k] - reportado_a5[k]) < 0.001 for k in reportado_a5),
          "Se reproducen exactamente las métricas de alerta reportadas en la A5 (recall 0.692, precisión 0.783, AUC 0.859)")

n = "Regla A5 (XGBoost regresión)"
ajustados[n] = v.ajustar(n, X_tr, y_tr, c_tr)
s_te[n] = v.puntuar(ajustados[n], X_te)
s_oof[n] = v.oof(n, v.fijo(n, ajustados[n]), X_tr, y_tr, c_tr, CV5)
sin_dur_15 = v.evaluar(y_te, s_te[n], 1.5)
print(f"Sin duración real, misma alerta (razón > 1.5): recall {sin_dur_15['recall']:.3f}, precisión {sin_dur_15['precision']:.3f}, AUC {sin_dur_15['roc_auc']:.3f}")

## 9. Entrenamiento de clasificadores

In [ ]:
for n in v.CLASIFICADORES:
    t0 = time.time()
    ajustados[n] = v.ajustar(n, X_tr, y_tr, c_tr)
    verificar(list(ajustados[n].classes_) == [0, 1], f"{n}: orden de clases [0, 1] (columna 1 = riesgo)")
    s_te[n] = v.puntuar(ajustados[n], X_te)
    s_oof[n] = v.oof(n, v.fijo(n, ajustados[n]), X_tr, y_tr, c_tr, CV5)
    print(f"{n}: mejores hiperparámetros {ajustados[n].best_params_} | AP interna {ajustados[n].best_score_:.3f} | {time.time() - t0:.0f} s")
verificar(all(len(s_te[m]) == 75 and len(s_oof[m]) == 297 for m in v.MODELOS), "Una puntuación por obra de prueba (75) y de entrenamiento (297) en los 5 modelos")

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 4.6))
ax.plot([0, 1], [0, 1], ls="--", color="#8a8a85", lw=1, label="Calibración perfecta")
calib = {}
for n in v.CLASIFICADORES:
    fr, pm = calibration_curve(y_tr, s_oof[n], n_bins=5, strategy="quantile")
    calib[n] = round(float(brier_score_loss(y_tr, s_oof[n])), 4)
    ax.plot(pm, fr, "o-", color=COLORES[n], lw=2, label=f"{n} (Brier {calib[n]:.3f})")
calib["Dummy (baseline)"] = round(float(brier_score_loss(y_tr, s_oof["Dummy (baseline)"])), 4)
ax.set_xlabel("Probabilidad media predicha"); ax.set_ylabel("Fracción observada de riesgo"); ax.set_title("Calibración (OOF, 5 grupos por cuantil)")
ax.legend(fontsize=8, loc="upper left"); fig.tight_layout(); guardar(fig, "03_calibracion")
print("Brier (menor es mejor):", calib)

## 10. Matrices de confusión

In [ ]:
tablas_u, umbral = {}, {"Dummy (baseline)": 0.50}
for n in v.MODELOS[1:]:
    tablas_u[n] = v.tabla_umbrales(y_tr, s_oof[n], v.rejilla(n))
    umbral[n] = v.elegir_umbral(tablas_u[n])["umbral"]
print("Umbrales de decisión elegidos (OOF entrenamiento):", umbral)

def dibujar_matrices(puntajes, umbrales, titulo, archivo):
    fig, axes = plt.subplots(1, len(puntajes), figsize=(3.3 * len(puntajes), 3.4))
    for ax, n in zip(axes, puntajes):
        vn, fp, fn, vp = v.conteos(y_te, (puntajes[n] >= umbrales[n]).astype(int))
        m = np.array([[vn, fp], [fn, vp]])
        ax.imshow(m, cmap="Blues", vmin=0, vmax=len(y_te))
        for (i, j), et in zip([(0, 0), (0, 1), (1, 0), (1, 1)], ["VN", "FP", "FN", "VP"]):
            ax.text(j, i, f"{et}\n{m[i, j]}", ha="center", va="center", fontsize=11, fontweight="bold",
                    color="white" if m[i, j] > len(y_te) * 0.45 else "#1a1a19")
        ax.set_xticks([0, 1], ["Sin alerta", "Alerta"]); ax.set_yticks([0, 1], ["Real 0", "Real 1"]); ax.grid(False)
        esc = "razón" if "Regla" in n else "p"
        ax.set_title(f"{CORTO[n]}\n({esc} ≥ {umbrales[n]:.2f})", fontsize=10)
    fig.suptitle(titulo, fontweight="bold"); fig.tight_layout(); guardar(fig, archivo)

verificar(all(sum(v.conteos(y_te, (s_te[n] >= umbral[n]).astype(int))) == 75 for n in v.MODELOS), "Cada matriz es 2×2 y suma 75 obras de prueba")
dibujar_matrices(s_te, umbral, "Matrices de confusión en prueba (75 obras, 26 con riesgo)", "04_matrices_confusion")

## 11. Cálculo de métricas

In [ ]:
filas = []
for n in v.MODELOS:
    e = v.evaluar(y_te, s_te[n], umbral[n])
    filas.append({"modelo": n, **{k: e[k] for k in ["umbral", "VP", "FP", "FN", "VN", "accuracy", "precision", "recall", "specificity",
                                                     "f1", "balanced_accuracy", "roc_auc", "average_precision", "alertas", "pct_alertas"]}})
metricas_test = pd.DataFrame(filas).set_index("modelo")
verificar(True, "Las 8 métricas por fórmula coinciden con scikit-learn en los 5 modelos (aserciones en v.evaluar)")
metricas_test.round(3).to_csv(TAB / "metricas_prueba.csv")
metricas_oof = pd.DataFrame({n: v.evaluar(y_tr, s_oof[n], umbral[n]) for n in v.MODELOS}).T
metricas_oof.round(3).to_csv(TAB / "metricas_oof_entrenamiento.csv")
print("Prueba (75 obras):"); display(metricas_test.round(3))
print("Validación OOF en entrenamiento (297 obras):"); display(metricas_oof[["umbral", "recall", "precision", "f1", "roc_auc", "average_precision", "pct_alertas"]].round(3))

## 12. ROC y AUC

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
for a, (yy, ss, tit) in zip(ax, [(y_tr, s_oof, "(a) Validación OOF – entrenamiento (297)"), (y_te, s_te, "(b) Prueba (75)")]):
    for n in v.MODELOS:
        fpr, tpr, _ = roc_curve(yy, ss[n])
        a.plot(fpr, tpr, color=COLORES[n], lw=2 if n != "Dummy (baseline)" else 1, label=f"{CORTO[n]} (AUC {roc_auc_score(yy, ss[n]):.3f})")
    a.plot([0, 1], [0, 1], ls="--", color="#c3c2b7", lw=1, label="Aleatorio (0.500)")
    a.set_xlabel("Tasa de falsos positivos (1 − especificidad)"); a.set_ylabel("Tasa de verdaderos positivos (recall)")
    a.set_title(tit); a.legend(loc="lower right", fontsize=8)
fig.suptitle("Curvas ROC consolidadas", fontweight="bold"); fig.tight_layout(); guardar(fig, "05_roc")

## 13. Curvas precisión-recall

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
for a, (yy, ss, tit) in zip(ax, [(y_tr, s_oof, "(a) Validación OOF – entrenamiento (297)"), (y_te, s_te, "(b) Prueba (75)")]):
    for n in v.MODELOS[1:]:
        p, r, _ = precision_recall_curve(yy, ss[n])
        a.plot(r, p, color=COLORES[n], lw=2, label=f"{CORTO[n]} (AP {average_precision_score(yy, ss[n]):.3f})")
    a.axhline(yy.mean(), ls="--", color="#8a8a85", lw=1, label=f"Aleatorio / Dummy = prevalencia ({yy.mean():.3f})")
    a.axvline(v.RECALL_MINIMO, ls=":", color="#c0392b", lw=1)
    a.set_xlabel("Recall"); a.set_ylabel("Precisión"); a.set_ylim(0, 1.03); a.set_title(tit); a.legend(loc="lower left", fontsize=8)
fig.suptitle("Curvas precisión-recall (línea roja: recall mínimo 0.80)", fontweight="bold"); fig.tight_layout(); guardar(fig, "06_precision_recall")

## 14. Análisis de umbrales

In [ ]:
for n in v.MODELOS[1:]:
    t = tablas_u[n]
    vista_u = [round(x, 2) for x in (np.arange(1.1, 2.01, 0.1) if "Regla" in n else np.arange(0.1, 0.91, 0.1))] + [umbral[n]]
    vista = t[t.umbral.round(2).isin(vista_u)].copy()
    vista["elegido"] = np.where(np.isclose(vista.umbral, umbral[n]), "◄", "")
    vista.round(3).to_csv(TAB / f"umbrales_{CORTO[n].replace(' ', '_').replace('.', '')}.csv", index=False)
    print(f"\n{n} — OOF entrenamiento"); display(vista.round(3))

fig, axes = plt.subplots(2, 2, figsize=(12, 7.6))
for a, n in zip(axes.ravel(), v.MODELOS[1:]):
    t = tablas_u[n]
    a.plot(t.umbral, t.recall, color="#eb6834", lw=2, label="Recall")
    a.plot(t.umbral, t.precision, color="#2a78d6", lw=2, label="Precisión")
    a.plot(t.umbral, t.f1, color="#1baf7a", lw=2, label="F1")
    a.plot(t.umbral, t.pct_alertas / 100, color="#8a8a85", lw=1.5, ls="--", label="% de obras con alerta")
    a.axhline(v.RECALL_MINIMO, color="#c0392b", lw=0.8, ls=":")
    a.axvline(umbral[n], color="#1a1a19", lw=1)
    a.text(umbral[n], 1.02, f" elegido {umbral[n]:.2f}", fontsize=9, va="bottom")
    a.set_ylim(0, 1.0); a.set_title(n, pad=14)
    a.set_xlabel("Razón costo estimado / preliminar" if "Regla" in n else "Umbral de probabilidad")
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h + [plt.Line2D([], [], color="#c0392b", lw=0.8, ls=":")], l + ["Recall mínimo 0.80"], loc="lower center", ncol=5, fontsize=9)
fig.suptitle("Sensibilidad al umbral de decisión (OOF entrenamiento)", fontweight="bold")
fig.tight_layout(rect=(0, 0.05, 1, 1)); guardar(fig, "07_umbrales")

comp_u = []
for n in v.MODELOS[1:]:
    defecto = 1.5 if "Regla" in n else 0.5
    for u_, et in [(defecto, "por defecto"), (umbral[n], "elegido (OOF)")]:
        e = v.evaluar(y_te, s_te[n], u_)
        comp_u.append({"modelo": n, "criterio": et, "umbral": u_, **{k: round(e[k], 3) for k in ["recall", "precision", "f1", "FP", "FN", "alertas"]}})
comp_u = pd.DataFrame(comp_u); comp_u.to_csv(TAB / "umbral_defecto_vs_elegido_prueba.csv", index=False); display(comp_u)

## 15. Validación cruzada

In [ ]:
filas_cv = []
t0 = time.time()
for k, (tr, va) in enumerate(CV5.split(X_tr, y_tr), start=1):
    Xa, Xv, ya, yv, ca = X_tr.iloc[tr], X_tr.iloc[va], y_tr.iloc[tr], y_tr.iloc[va], c_tr.iloc[tr]
    interna = StratifiedKFold(5, shuffle=True, random_state=SEMILLA + k)
    for n in v.MODELOS:
        m = v.ajustar(n, Xa, ya, ca, semilla_cv=SEMILLA + k)
        s = v.puntuar(m, Xv)
        if n == "Dummy (baseline)":
            u = 0.5
        else:
            u = v.elegir_umbral(v.tabla_umbrales(ya, v.oof(n, v.fijo(n, m), Xa, ya, ca, interna), v.rejilla(n)))["umbral"]
        e = v.evaluar(yv, s, u)
        filas_cv.append({"pliegue": k, "modelo": n, "umbral": u, **{c: e[c] for c in ["recall", "precision", "f1", "roc_auc", "average_precision", "pct_alertas"]}})
cv_anidada = pd.DataFrame(filas_cv)
cv_anidada.round(4).to_csv(TAB / "cv_anidada_por_pliegue.csv", index=False)
MET = ["average_precision", "roc_auc", "recall", "precision", "f1", "pct_alertas"]
cv_res = cv_anidada.groupby("modelo", sort=False)[MET].agg(["mean", "std"]).round(3)
cv_res.to_csv(TAB / "cv_anidada_resumen.csv")
print(f"Validación anidada: {time.time() - t0:.0f} s"); display(cv_res)
display(cv_anidada.pivot(index="pliegue", columns="modelo", values="average_precision").round(3))

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3.8))
for a, met, tit in zip(ax, ["average_precision", "roc_auc", "recall", "precision"], ["Average precision", "ROC-AUC", "Recall", "Precisión"]):
    for i, n in enumerate(v.MODELOS):
        vals = cv_anidada.loc[cv_anidada.modelo == n, met].values
        a.scatter(np.full(5, i) + np.linspace(-0.12, 0.12, 5), vals, s=28, color=COLORES[n], zorder=3)
        a.plot([i - 0.3, i + 0.3], [vals.mean()] * 2, color="#1a1a19", lw=2)
    a.set_xticks(range(5), ["Dummy", "Regla A5", "Reg. log.", "RF", "XGB"], fontsize=8.5); a.set_title(tit); a.set_ylim(-0.03, 1.03)
ax[2].axhline(v.RECALL_MINIMO, color="#c0392b", lw=0.8, ls=":"); ax[3].axhline(0.70, color="#c0392b", lw=0.8, ls=":")
fig.suptitle("Validación cruzada anidada (5 pliegues; línea negra = media; punteado = meta)", fontweight="bold")
fig.tight_layout(); guardar(fig, "08_validacion_cruzada")

### Evaluación cronológica

In [ ]:
orden = df.sort_values(["indice_tiempo_inicio", "id_proyecto"]).index
corte = int(round(len(orden) * 0.8))
it_tr, it_te = orden[:corte], orden[corte:]
verificar(df.loc[it_tr, "indice_tiempo_inicio"].max() <= df.loc[it_te, "indice_tiempo_inicio"].min(), "Partición cronológica: todo el entrenamiento inicia antes o igual que la prueba")
filas_t = []
for n in v.MODELOS:
    m = v.ajustar(n, X.loc[it_tr], y.loc[it_tr], df.loc[it_tr, "costo_real_m2"])
    s = v.puntuar(m, X.loc[it_te])
    u = 0.5 if n == "Dummy (baseline)" else v.elegir_umbral(v.tabla_umbrales(
        y.loc[it_tr], v.oof(n, v.fijo(n, m), X.loc[it_tr], y.loc[it_tr], df.loc[it_tr, "costo_real_m2"], CV5), v.rejilla(n)))["umbral"]
    e = v.evaluar(y.loc[it_te], s, u)
    filas_t.append({"modelo": n, "umbral": u, **{c: round(e[c], 3) for c in ["roc_auc", "average_precision", "recall", "precision", "pct_alertas"]}})
temporal = pd.DataFrame(filas_t).set_index("modelo")
temporal.to_csv(TAB / "evaluacion_cronologica.csv")
print(f"Entrenamiento: años {df.loc[it_tr, 'anio_inicio'].min()}–{df.loc[it_tr, 'anio_inicio'].max()} (prevalencia {y.loc[it_tr].mean():.3f}); "
      f"prueba: {df.loc[it_te, 'anio_inicio'].min()}–{df.loc[it_te, 'anio_inicio'].max()} (prevalencia {y.loc[it_te].mean():.3f}, {int(y.loc[it_te].sum())} positivas)")
display(temporal)

## 16. Comparación estadística

In [ ]:
RCV = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEMILLA)
fijos = {n: v.fijo(n, ajustados[n]) for n in v.MODELOS}
filas_r = []
t0 = time.time()
for j, (tr, va) in enumerate(RCV.split(X_tr, y_tr)):
    for n in v.MODELOS:
        mf = v.ReglaRegresionA5(fijos[n].params) if "Regla" in n else clone(fijos[n])
        m = v.ajustar_fijo(n, mf, X_tr.iloc[tr], y_tr.iloc[tr], c_tr.iloc[tr])
        s = v.puntuar(m, X_tr.iloc[va])
        filas_r.append({"particion": j, "modelo": n, "average_precision": average_precision_score(y_tr.iloc[va], s),
                        "roc_auc": roc_auc_score(y_tr.iloc[va], s)})
rcv = pd.DataFrame(filas_r)
verificar(rcv.groupby("modelo").size().eq(50).all(), "Los 5 modelos se evaluaron en las mismas 50 particiones")
rcv.round(5).to_csv(TAB / "cv_repetida_5x10.csv", index=False)
print(f"CV repetida: {time.time() - t0:.0f} s")
display(rcv.groupby("modelo", sort=False)[["average_precision", "roc_auc"]].agg(["mean", "std"]).round(3))

from itertools import combinations
comp = []
for met in ["average_precision", "roc_auc"]:
    piv = rcv.pivot(index="particion", columns="modelo", values=met)
    for a_, b_ in combinations(v.MODELOS, 2):
        d = piv[b_] - piv[a_]
        t, p = v.t_corregida(piv[b_], piv[a_], frac_prueba=0.2)
        li, ls = v.ic_t_corregida(d, 0.2)
        comp.append({"metrica": met, "A": a_, "B": b_, "media_A": piv[a_].mean(), "media_B": piv[b_].mean(), "dif_B_menos_A": d.mean(),
                     "ic95_inf": li, "ic95_sup": ls, "t_corregida": t, "p": p, "d_pareada": d.mean() / d.std(ddof=1) if d.std() > 0 else np.nan,
                     "B_gana": int((d > 0).sum())})
comp = pd.DataFrame(comp)
for met in ["average_precision", "roc_auc"]:
    sel = comp.metrica == met
    ajust = v.holm(dict(zip(comp.index[sel], comp.loc[sel, "p"])))
    comp.loc[sel, "p_holm"] = [ajust[i] for i in comp.index[sel]]
comp["significativa"] = np.where(comp.p_holm < 0.05, "Sí", "No")
comp.round(5).to_csv(TAB / "comparaciones_estadisticas.csv", index=False)
display(comp[comp.metrica == "average_precision"].round(4))

In [ ]:
ap_cv = cv_res.loc[v.CLASIFICADORES, ("average_precision", "mean")]
mejor = ap_cv.idxmax()
def sig(a_, b_, met="average_precision"):
    f = comp[(comp.metrica == met) & (((comp.A == a_) & (comp.B == b_)) | ((comp.A == b_) & (comp.B == a_)))].iloc[0]
    return f
seleccion_log = []
SELECCIONADO = mejor
for simple in v.CLASIFICADORES:
    if simple == mejor:
        break
    f = sig(simple, mejor)
    seleccion_log.append(f"{mejor} vs {simple}: Δ AP = {(f.media_B - f.media_A) * (1 if f.B == mejor else -1):+.3f}, p Holm = {f.p_holm:.3g}")
    if f.p_holm >= 0.05:
        SELECCIONADO = simple
        seleccion_log.append(f"→ sin diferencia significativa: se elige el más simple, {simple}")
        break
print("Mayor AP anidada:", mejor, "| seleccionado:", SELECCIONADO); print("\n".join(seleccion_log))
U_SEL = umbral[SELECCIONADO]
REGLA = "Regla A5 (XGBoost regresión)"

pred_sel = (s_te[SELECCIONADO] >= U_SEL).astype(int); pred_regla = (s_te[REGLA] >= umbral[REGLA]).astype(int)
mcn = v.mcnemar_exacta(y_te, pred_regla, pred_sel)
boot = {"roc_auc": v.bootstrap_diferencia(y_te, s_te[REGLA], s_te[SELECCIONADO], roc_auc_score),
        "average_precision": v.bootstrap_diferencia(y_te, s_te[REGLA], s_te[SELECCIONADO], average_precision_score)}
prueba_extra = {"mcnemar_seleccionado_vs_regla": mcn, "bootstrap_prueba_seleccionado_menos_regla": boot}
print(json.dumps(prueba_extra, indent=1, ensure_ascii=False))

claves = [(a_, b_) for a_, b_ in [(REGLA, SELECCIONADO), ("Dummy (baseline)", SELECCIONADO), ("Dummy (baseline)", REGLA)]] + \
         [(SELECCIONADO, o) if v.MODELOS.index(SELECCIONADO) < v.MODELOS.index(o) else (o, SELECCIONADO) for o in v.CLASIFICADORES if o != SELECCIONADO]
fig, ax = plt.subplots(figsize=(9, 3.9))
for i, (a_, b_) in enumerate(claves):
    f = sig(a_, b_)
    dd = (rcv[rcv.modelo == f.B].sort_values("particion").average_precision.values - rcv[rcv.modelo == f.A].sort_values("particion").average_precision.values)
    ax.scatter(dd, np.full(50, i) + np.random.default_rng(i).uniform(-0.13, 0.13, 50), s=11, color="#86b6ef", zorder=2)
    ax.plot([f.ic95_inf, f.ic95_sup], [i, i], color="#1a1a19", lw=3, zorder=3); ax.plot(f.dif_B_menos_A, i, "o", color="#1a1a19", ms=7, zorder=4)
    ax.text(max(f.ic95_sup, dd.max()) + 0.01, i, f"Δ={f.dif_B_menos_A:+.3f}  p Holm={f.p_holm:.2g}", va="center", fontsize=8.5)
ax.axvline(0, color="#8a8a85", ls="--", lw=1)
ax.set_yticks(range(len(claves)), [f"{CORTO[sig(a_, b_).B]} − {CORTO[sig(a_, b_).A]}" for a_, b_ in claves]); ax.invert_yaxis()
ax.set_xlabel("Diferencia pareada de average precision (50 particiones); barra = IC 95 % corregido")
ax.set_title("Comparación estadística (t corregida de Nadeau y Bengio + Holm)"); ax.set_xlim(None, ax.get_xlim()[1] + 0.2)
fig.tight_layout(); guardar(fig, "09_comparacion_estadistica")

## 17. Diseño y simulación A/B

In [ ]:
SIM = "SIMULACIÓN"
oof_A = v.evaluar(y_tr, s_oof[REGLA], umbral[REGLA]); oof_B = v.evaluar(y_tr, s_oof[SELECCIONADO], U_SEL)
pA, pB = oof_A["precision"], oof_B["precision"]
tasa_A, tasa_B = oof_A["pct_alertas"] / 100, oof_B["pct_alertas"] / 100
if pB > pA:
    n_alertas = v.n_por_grupo_proporciones(pA, pB)
    n_obras = int(np.ceil(n_alertas / min(tasa_A, tasa_B)))
else:
    n_alertas = n_obras = None
print(f"Precisión esperada (OOF): A = {pA:.3f}, B = {pB:.3f}; alertas por grupo requeridas = {n_alertas}; obras por grupo ≈ {n_obras}")

def simular(n_obras_grupo, rng):
    ia, ib = rng.integers(0, 75, n_obras_grupo), rng.integers(0, 75, n_obras_grupo)
    ya, yb = y_te.values[ia], y_te.values[ib]
    aa, ab = pred_regla[ia], pred_sel[ib]
    xa, na, xb, nb = int((aa & ya).sum()), int(aa.sum()), int((ab & yb).sum()), int(ab.sum())
    p1, p2, pp = xa / na, xb / nb, (xa + xb) / (na + nb)
    z = (p2 - p1) / np.sqrt(pp * (1 - pp) * (1 / na + 1 / nb))
    return {"precision_A": p1, "precision_B": p2, "recall_A": xa / max(ya.sum(), 1), "recall_B": xb / max(yb.sum(), 1),
            "alertas_A": na, "alertas_B": nb, "z": z, "p_unilateral": float(stats.norm.sf(z))}
def simular_diseno(n_obras_grupo, rng):
    na, nb = rng.binomial(n_obras_grupo, tasa_A), rng.binomial(n_obras_grupo, tasa_B)
    xa, xb = rng.binomial(na, pA), rng.binomial(nb, pB)
    p1, p2, pp = xa / na, xb / nb, (xa + xb) / (na + nb)
    z = (p2 - p1) / np.sqrt(pp * (1 - pp) * (1 / na + 1 / nb))
    return {"precision_A": p1, "precision_B": p2, "alertas_A": na, "alertas_B": nb, "z": z, "p_unilateral": float(stats.norm.sf(z))}
rng = np.random.default_rng(SEMILLA)
N_SIM = n_obras or 500
escenarios = {}
for nombre_e, f in [("Supuesto de diseño (tasas OOF)", simular_diseno), ("Remuestreo de las 75 obras de prueba", simular)]:
    ejemplo = f(N_SIM, rng)
    reps = pd.DataFrame([f(N_SIM, rng) for _ in range(2000)])
    escenarios[nombre_e] = {"ejemplo": ejemplo, "potencia": float((reps.p_unilateral < 0.05).mean()),
                            "precision_A_media": float(reps.precision_A.mean()), "precision_B_media": float(reps.precision_B.mean()),
                            "prob_B_peor_que_A": float((reps.precision_B < reps.precision_A).mean())}
    print(f"[{SIM}] {nombre_e}: {N_SIM} obras por grupo | una corrida: precisión A {ejemplo['precision_A']:.3f} vs B {ejemplo['precision_B']:.3f}, "
          f"z = {ejemplo['z']:.2f}, p = {ejemplo['p_unilateral']:.4f} | potencia en 2,000 repeticiones {escenarios[nombre_e]['potencia']:.1%}")
ab = {"precision_esperada_A": pA, "precision_esperada_B": pB, "recall_esperado_A": oof_A["recall"], "recall_esperado_B": oof_B["recall"],
      "tasa_alertas_A": tasa_A, "tasa_alertas_B": tasa_B, "alertas_por_grupo": n_alertas, "obras_por_grupo": n_obras, "escenarios": escenarios}
pd.DataFrame([{"escenario": k, **{c: x for c, x in e.items() if c != "ejemplo"}, **{f"ejemplo_{c}": x for c, x in e["ejemplo"].items()}}
              for k, e in escenarios.items()]).to_csv(TAB / "ab_simulacion.csv", index=False)

## 18. Impacto sobre VOREKO

In [ ]:
filas_s = []
for u_s in (0.2, 0.3, 0.4, 0.5):
    yy = v.construir_etiqueta(df, u_s)
    for n in [REGLA, SELECCIONADO]:
        mf = v.ReglaRegresionA5(fijos[n].params) if "Regla" in n else clone(fijos[n])
        so = v.oof(n, mf, X_tr, yy.loc[idx_tr], c_tr, CV5)
        mt = v.ajustar_fijo(n, v.ReglaRegresionA5(fijos[n].params) if "Regla" in n else clone(fijos[n]), X_tr, yy.loc[idx_tr], c_tr)
        st = v.puntuar(mt, X_te)
        filas_s.append({"umbral_sobrecosto": f"{int(u_s * 100)} %", "prevalencia": round(yy.mean(), 3), "modelo": n,
                        "AUC_OOF": round(roc_auc_score(yy.loc[idx_tr], so), 3), "AP_OOF": round(average_precision_score(yy.loc[idx_tr], so), 3),
                        "AUC_prueba": round(roc_auc_score(yy.loc[idx_te], st), 3), "AP_prueba": round(average_precision_score(yy.loc[idx_te], st), 3)})
sens_etiqueta = pd.DataFrame(filas_s); sens_etiqueta.to_csv(TAB / "sensibilidad_umbral_sobrecosto.csv", index=False); display(sens_etiqueta)

### Integración funcional

In [ ]:
t_sel = tablas_u[SELECCIONADO]
U_AMARILLO = float(t_sel[t_sel.recall >= 0.95].umbral.max())
def nivel(p): return np.where(p >= U_SEL, "Rojo", np.where(p >= U_AMARILLO, "Amarillo", "Verde"))
semaforo = []
for nombre_conj, (yy, ss) in {"OOF entrenamiento": (y_tr, s_oof[SELECCIONADO]), "Prueba": (y_te, s_te[SELECCIONADO])}.items():
    niv = pd.Series(nivel(ss), index=yy.index)
    for k in ["Verde", "Amarillo", "Rojo"]:
        semaforo.append({"conjunto": nombre_conj, "nivel": k, "obras": int((niv == k).sum()), "con_riesgo_real": int(yy[niv == k].sum()),
                         "tasa_riesgo_observada": round(float(yy[niv == k].mean()), 3) if (niv == k).any() else None})
semaforo = pd.DataFrame(semaforo); semaforo.to_csv(TAB / "semaforo_propuesto.csv", index=False)
print(f"Umbral amarillo (recall OOF ≥ 0.95): {U_AMARILLO:.2f} | umbral rojo: {U_SEL:.2f}"); display(semaforo)

modelo_final = ajustados[SELECCIONADO].best_estimator_
i_ej = int(np.argmax(s_te[SELECCIONADO]))
obra = X_te.iloc[[i_ej]]
factores = v.factores_locales(modelo_final, obra, X_tr, top=3)
salida_ejemplo = {"id_proyecto": df.loc[X_te.index[i_ej], "id_proyecto"], "version_modelo": f"{v.NOMBRE_REGISTRO} v1 (candidato)",
                  "fecha_evaluacion": FECHA[:10], "puntuacion_riesgo": round(float(s_te[SELECCIONADO][i_ej]), 3), "umbral": U_SEL,
                  "clasificacion": int(s_te[SELECCIONADO][i_ej] >= U_SEL), "nivel": str(nivel(s_te[SELECCIONADO][[i_ej]])[0]),
                  "factores": [{"variable": f["variable"], "valor": round(f["valor"], 3), "efecto_en_puntuacion": round(f["efecto"], 3)} for f in factores],
                  "recomendacion": "Revisar partidas de mayor peso, actualizar precios unitarios y definir contingencia antes de iniciar."}
print(json.dumps(salida_ejemplo, indent=1, ensure_ascii=False))

from sklearn.inspection import permutation_importance
pi = permutation_importance(modelo_final, X_te, y_te, scoring="average_precision", n_repeats=30, random_state=SEMILLA)
imp_glob = pd.DataFrame({"variable": X_te.columns, "media": pi.importances_mean, "std": pi.importances_std}).sort_values("media", ascending=False)
imp_glob.round(4).to_csv(TAB / "importancia_permutacion.csv", index=False)
top = imp_glob.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top.variable, top.media, xerr=top["std"], color=COLORES[SELECCIONADO], height=0.6, error_kw={"ecolor": "#8a8a85", "lw": 1})
ax.axvline(0, color="#8a8a85", lw=0.8)
ax.set_xlabel("Caída media de average precision al permutar (30 repeticiones)"); ax.set_title(f"{SELECCIONADO}: variables más influyentes (prueba)")
fig.tight_layout(); guardar(fig, "11_importancia_variables")

impacto = []
for n in [REGLA, SELECCIONADO]:
    e = metricas_test.loc[n]
    impacto.append({"sistema": n, "alertas_por_100": round(e.alertas / 75 * 100, 1), "riesgos_detectados_por_100": round(e.VP / 75 * 100, 1),
                    "riesgos_omitidos_por_100": round(e.FN / 75 * 100, 1), "alertas_falsas_por_100": round(e.FP / 75 * 100, 1)})
impacto = pd.DataFrame(impacto); impacto.to_csv(TAB / "impacto_por_100_obras.csv", index=False); display(impacto)

## 19. Selección final

In [ ]:
e_sel = metricas_test.loc[SELECCIONADO]
f_dummy = sig("Dummy (baseline)", SELECCIONADO)
smart = pd.DataFrame([
    {"criterio": "Recall de la clase de riesgo ≥ 0.80 (prueba)", "resultado": round(e_sel.recall, 3), "cumple": e_sel.recall >= 0.80},
    {"criterio": "Precisión ≥ 0.70 (prueba)", "resultado": round(e_sel.precision, 3), "cumple": e_sel.precision >= 0.70},
    {"criterio": "AP superior al baseline (CV repetida, p Holm < 0.05)", "resultado": round(f_dummy.p_holm, 6), "cumple": f_dummy.p_holm < 0.05},
    {"criterio": "Validación cruzada estratificada realizada", "resultado": "5 pliegues anidada + 5×10 repetida", "cumple": True},
    {"criterio": "Evaluación independiente en prueba", "resultado": "75 obras, hold-out A5", "cumple": True}])
smart["cumple"] = np.where(smart.cumple, "Sí", "No"); smart.to_csv(TAB / "verificacion_smart.csv", index=False)
print(f"Modelo seleccionado: {SELECCIONADO} | umbral {U_SEL:.2f}"); display(smart)

fig, ax = plt.subplots(figsize=(10, 3.8))
mets = ["recall", "precision", "f1", "balanced_accuracy", "roc_auc", "average_precision"]
etq = ["Recall", "Precisión", "F1", "Exactitud\nbalanceada", "ROC-AUC", "Average\nprecision"]
xx = np.arange(len(mets)); w = 0.16
for i, n in enumerate(v.MODELOS):
    ax.bar(xx + (i - 2) * w, metricas_test.loc[n, mets].astype(float), width=w - 0.02, color=COLORES[n], label=CORTO[n])
ax.axhline(0.80, color="#c0392b", lw=0.8, ls=":"); ax.axhline(0.70, color="#c0392b", lw=0.8, ls=":")
ax.set_xticks(xx, etq); ax.set_ylim(0, 1); ax.legend(ncol=5, fontsize=8, loc="upper left")
ax.set_title("Métricas en prueba con el umbral elegido en validación (punteado: metas 0.80 y 0.70)")
fig.tight_layout(); guardar(fig, "10_resumen_metricas_prueba")

## 20. Registro de experimentos (MLflow)

In [ ]:
import subprocess
def commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=v.RAIZ_A6, stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "sin-git"
mlflow.set_tracking_uri(f"sqlite:///{(v.RAIZ_A6 / 'mlflow.db').as_posix()}")
if mlflow.get_experiment_by_name(v.EXPERIMENTO) is None:
    mlflow.create_experiment(v.EXPERIMENTO, artifact_location=(v.RAIZ_A6 / "mlruns").as_uri(),
                             tags={"proyecto": "VOREKO IA", "problema": "clasificación riesgo de sobrecosto > 50 %"})
mlflow.set_experiment(v.EXPERIMENTO)
TAGS = {"proyecto": "VOREKO IA", "actividad": "6", "autor": "José Ignacio Castillo Camacho", "git_commit": commit(),
        "version_datos": man_clasif["version_datos"], "sha256_csv_limpio_A5": manifiesto_a5["huellas_sha256"]["csv_limpio"],
        "sha256_csv_clasificacion": man_clasif["sha256_csv_clasificacion"], "relacion": "evoluciona voreko_costo_real_m2 v1 (A5, regresión)",
        **{f"v_{k}": val for k, val in VERSIONES.items()}}
runs = {}
with mlflow.start_run(run_name="01_datos_clasificacion", tags={**TAGS, "etapa": "datos"}) as r:
    mlflow.log_input(mlflow.data.from_pandas(pd.read_csv(v.CSV_CLASIF), source=str(v.CSV_CLASIF), name="residential_building_clasificacion",
                                             targets="riesgo_sobrecosto"), context="clasificacion")
    mlflow.log_params({"filas": len(df), "predictoras": X.shape[1], "excluidas_a6": ",".join(v.EXCLUIDAS_A6), "umbral_sobrecosto": 0.5,
                       "particion": "hold-out A5 80/20 (quintiles costo real, semilla 42)", "positivos": int(y.sum())})
    mlflow.log_metrics({"prevalencia": float(y.mean()), "obras_en_50pct": empates})
    mlflow.log_artifact(str(v.MANIFIESTO_CLASIF), artifact_path="datos"); runs["datos"] = r.info.run_id
for n in v.MODELOS:
    with mlflow.start_run(run_name=f"02_modelo_{CORTO[n].replace(' ', '_')}", tags={**TAGS, "etapa": "modelo", "modelo": n}) as r:
        params = getattr(ajustados[n], "best_params_", None) or (ajustados[n].params if "Regla" in n else {"strategy": "prior"})
        mlflow.log_params({**{k.split("__")[-1]: str(val) for k, val in params.items()}, "umbral_decision": umbral[n],
                           "escala_umbral": "razón costo estimado/preliminar" if "Regla" in n else "probabilidad",
                           "regla_umbral": "máx. precisión con recall ≥ 0.80 (OOF)"})
        mlflow.log_metrics({**{f"test_{k}": float(metricas_test.loc[n, k]) for k in ["recall", "precision", "f1", "specificity", "accuracy", "balanced_accuracy", "roc_auc", "average_precision"]},
                            **{f"cv_{m}_{s}": float(cv_res.loc[n, (m, s)]) for m in MET for s in ("mean", "std")},
                            **{f"oof_{k}": float(metricas_oof.loc[n, k]) for k in ["recall", "precision", "roc_auc", "average_precision"]}})
        if n in tablas_u:
            for _, f in tablas_u[n].iloc[::5].iterrows():
                mlflow.log_metrics({"umbral_recall": f.recall, "umbral_precision": f.precision}, step=int(round(f.umbral * 100)))
        runs[n] = r.info.run_id
with mlflow.start_run(run_name="03_validacion_y_comparacion", tags={**TAGS, "etapa": "comparacion"}) as r:
    mlflow.log_params({"cv_anidada": "StratifiedKFold(5) externa + GridSearchCV(5) interna", "cv_repetida": "RepeatedStratifiedKFold(5x10)",
                       "prueba": "t corregida Nadeau-Bengio + Holm; McNemar exacta; bootstrap 2000", "seleccionado": SELECCIONADO})
    mlflow.log_metrics({"mcnemar_p": mcn["p_exacta"], "boot_dif_auc": boot["roc_auc"]["diferencia"], "boot_dif_ap": boot["average_precision"]["diferencia"]})
    for f in sorted(TAB.glob("*.csv")) + sorted(FIG.glob("*.png")):
        mlflow.log_artifact(str(f), artifact_path="tablas" if f.suffix == ".csv" else "figuras")
    runs["comparacion"] = r.info.run_id
with mlflow.start_run(run_name=f"04_modelo_seleccionado_{CORTO[SELECCIONADO].replace(' ', '_')}", tags={**TAGS, "etapa": "seleccion"}) as r:
    mlflow.log_params({"modelo": SELECCIONADO, "umbral_decision": U_SEL, "umbral_amarillo": U_AMARILLO})
    from mlflow.models import infer_signature
    tipos = ["datos_prep.SelectorRezagos", "datos_prep.columnas_categoricas", "datos_prep.columnas_numericas", "sklearn.tree._tree.Tree",
             "xgboost.sklearn.XGBClassifier", "xgboost.core.Booster"]
    info = mlflow.sklearn.log_model(modelo_final, name="modelo", signature=infer_signature(X_tr, modelo_final.predict_proba(X_tr)[:, 1]),
                                    input_example=X_tr.head(3), code_paths=[str(Path(v.dp.__file__))], skops_trusted_types=tipos)
    runs["seleccionado"] = r.info.run_id
mv = mlflow.register_model(info.model_uri, v.NOMBRE_REGISTRO)
cli = mlflow.MlflowClient()
cli.set_registered_model_alias(v.NOMBRE_REGISTRO, "candidato", mv.version)
cli.set_model_version_tag(v.NOMBRE_REGISTRO, mv.version, "estado", "candidato: no promovido a producción")
cli.set_model_version_tag(v.NOMBRE_REGISTRO, mv.version, "umbral_decision", str(U_SEL))
corridas = mlflow.search_runs([mlflow.get_experiment_by_name(v.EXPERIMENTO).experiment_id])
verificar(len(corridas) >= 8, f"MLflow: {len(corridas)} runs en {v.EXPERIMENTO}; modelo registrado v{mv.version} con alias 'candidato'")
corridas[["run_id", "tags.mlflow.runName", "tags.etapa"]].to_csv(TAB / "mlflow_runs.csv", index=False)

## 21. Exportación de resultados

In [ ]:
def nativo(o):
    if isinstance(o, dict): return {str(k): nativo(x) for k, x in o.items()}
    if isinstance(o, (list, tuple)): return [nativo(x) for x in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, (np.floating, float)): return None if np.isnan(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
resumen = {
    "fecha_ejecucion": FECHA, "versiones": VERSIONES, "semilla": SEMILLA, "manifiesto_clasificacion": man_clasif,
    "duracion": dur, "calidad": calidad, "empates_50": empates, "positivos_sin_redondeo": flotante,
    "prevalencia_por_umbral": sens_prev.to_dict(orient="records"), "particion": particion.to_dict(),
    "historico_a5_reproducido": {k: hist[k] for k in ["recall", "precision", "roc_auc", "average_precision", "VP", "FP", "FN", "VN"]},
    "regla_a5_sin_duracion_umbral_15": {k: sin_dur_15[k] for k in ["recall", "precision", "roc_auc", "average_precision", "VP", "FP", "FN"]},
    "hiperparametros": {n: {k: str(x) for k, x in (getattr(ajustados[n], "best_params_", None) or (ajustados[n].params if "Regla" in n else {"strategy": "prior"})).items()} for n in v.MODELOS},
    "umbral": umbral, "brier_oof": calib,
    "metricas_prueba": metricas_test.to_dict(orient="index"), "metricas_oof": metricas_oof.to_dict(orient="index"),
    "umbral_defecto_vs_elegido": comp_u.to_dict(orient="records"),
    "cv_anidada": {n: {f"{m}_{s}": cv_res.loc[n, (m, s)] for m in MET for s in ("mean", "std")} for n in v.MODELOS},
    "cv_anidada_pliegues": cv_anidada.to_dict(orient="records"), "cronologica": temporal.reset_index().to_dict(orient="records"),
    "cronologica_info": {"anios_train": [int(df.loc[it_tr, "anio_inicio"].min()), int(df.loc[it_tr, "anio_inicio"].max())],
                         "anios_test": [int(df.loc[it_te, "anio_inicio"].min()), int(df.loc[it_te, "anio_inicio"].max())],
                         "prev_train": float(y.loc[it_tr].mean()), "prev_test": float(y.loc[it_te].mean()), "pos_test": int(y.loc[it_te].sum())},
    "cv_repetida": rcv.groupby("modelo")[["average_precision", "roc_auc"]].agg(["mean", "std"]).round(4).to_dict(),
    "comparaciones": comp.to_dict(orient="records"), "seleccion": {"mayor_ap": mejor, "seleccionado": SELECCIONADO, "log": seleccion_log, "umbral": U_SEL},
    "prueba_extra": prueba_extra, "ab": ab, "sensibilidad_etiqueta": sens_etiqueta.to_dict(orient="records"),
    "semaforo": {"umbral_amarillo": U_AMARILLO, "umbral_rojo": U_SEL, "tabla": semaforo.to_dict(orient="records")},
    "salida_ejemplo": salida_ejemplo, "impacto_100": impacto.to_dict(orient="records"), "importancia": imp_glob.head(10).to_dict(orient="records"), "smart": smart.to_dict(orient="records"),
    "mlflow": {"experimento": v.EXPERIMENTO, "runs": len(corridas), "registro": v.NOMBRE_REGISTRO, "version": int(mv.version), "alias": "candidato",
               "commit": TAGS["git_commit"]},
    "checks": CHECKS}
(RES / "resumen_resultados.json").write_text(json.dumps(nativo(resumen), ensure_ascii=False, indent=1), encoding="utf-8")
(RES / "pruebas_consistencia.txt").write_text("\n".join(f"OK  {c}" for c in CHECKS), encoding="utf-8")
print(f"Pruebas de consistencia superadas: {len(CHECKS)}")
for p in sorted(RES.rglob("*")):
    if p.is_file(): print(" ", p.relative_to(v.RAIZ_A6))
if EN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/resultados_A6", "zip", RES); files.download("/content/resultados_A6.zip")